In [ ]:
# DATA EXTRACTION AND CLEANING
#
# This notebook starts from the provisional application-level dataset
# produced by DataPreparation:
# - applications retrieved through keywords, IPC or CPC and retained by SBERT;
# - additional applications selected through semantic enrichment.
#
#
# In this notebook we will:
# 1. Retrieve all available members of the selected DOCDB families.
# 2. Build a family-level table and establish the earliest priority year.
# 3. Extract applicants and countries, preserving their links to applications.
# 4. Normalize applicant names using PATSTAT standardized names where available.
# 5. Retrieve classifications and citation records for subsequent analyses.
#
# Family expansion will not reapply the original keyword, classification,
# SBERT or application filing-year filters.

from pathlib import Path

import pandas as pd
from IPython.display import display

# Exported from DataPreparation using:
# df_selected.to_parquet("df_selected.parquet", index=False)
input_file = Path("df_selected.parquet")

if not input_file.is_file():
    raise FileNotFoundError(
        f"Dataset not found: {input_file.resolve()}. "
        "Export df_selected from DataPreparation first, "
        "or update input_file to its location."
    )

df_selected = pd.read_parquet(input_file)

required_columns = {"appln_id", "docdb_family_id", "retrieval_source"}
missing_columns = required_columns - set(df_selected.columns)

if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

if df_selected["appln_id"].isna().any():
    raise ValueError("The input contains missing application IDs.")

if df_selected["appln_id"].duplicated().any():
    raise ValueError("The input contains duplicate application IDs.")

print("PROVISIONAL INPUT DATASET")
print(f"Applications: {len(df_selected):,}")
print(f"DOCDB families: {df_selected['docdb_family_id'].nunique():,}")
print(f"Columns: {len(df_selected.columns)}")
print("Validation status: pending")

print("\nDATA PREVIEW")
display(df_selected.head())

# Show column names, data types, missing values and blank text values.
schema = pd.DataFrame({
    "column": df_selected.columns,
    "dtype": [str(df_selected[c].dtype) for c in df_selected.columns],
    "missing_values": [
        int(df_selected[c].isna().sum())
        for c in df_selected.columns
    ],
    "blank_strings": [
        int(df_selected[c].map(
            lambda value: isinstance(value, str) and not value.strip()
        ).sum())
        for c in df_selected.columns
    ],
})

print("\nCOLUMN OVERVIEW")
display(schema)

print("\nAPPLICATIONS BY RETRIEVAL SOURCE")
display(
    df_selected["retrieval_source"]
    .value_counts(dropna=False)
    .rename_axis("retrieval_source")
    .reset_index(name="applications")
)

# Keep the imported selection intact.
# Expanded family members and cleaned tables will use separate DataFrames.